# Semana 3: Arquitectura Medallion — Solución

**Instructor:** Luciano Argolo | lucianoargolo.com


## Ejercicio 1.1: Verificar catálogo y explorar Bronze

In [0]:
%sql
use catalog bootcamp;
show schemas;

In [0]:
%sql
select
count(*)
from bootcamp.bronze.properties_bronze

## Ejercicio 1.2: Crear esquemas Silver y Gold + verificar las 3 capas


In [0]:
%sql
-- ============================================================
-- PASO 2: Crear el esquema Silver
-- ============================================================

-- Crear esquema Silver

CREATE SCHEMA IF NOT EXISTS bootcamp.silver
COMMENT 'Capa Silver: Datos limpios, validados y con tipos correctos';

-- Crear esquema Golf
CREATE SCHEMA IF NOT EXISTS bootcamp.gold;


-- Verificar
SHOW SCHEMAS;

In [0]:
%sql
describe schema bootcamp.silver

In [0]:
%sql
describe schema bootcamp.gold

## Ejercicio 2.1: Diseñar estructura de tabla Silver

Columnas necesarias basadas en el EDA de Semana 2:

- **Ubicación:** `partido`, `region` (derivadas de `zona` en Bronze)
- **Precio:** `precio`, `moneda`, `expensas`, `precio_por_m2`
- **Características:** `ambientes`, `metros_cuadrados_totales`, `metros_cuadrados_cubiertos`, `antiguedad`, `cochera`
- **Metadata:** `_source_table`, `_processing_timestamp`


In [0]:
%sql
create table if not exists bootcamp.silver.propiedades (
    -- identificación
    propiedad_id               bigint        not null comment 'xxhash64 de la url: el mismo aviso siempre tiene el mismo id',
    id_bronze                  double        comment 'id del registro en bronze del que sale la última versión',
    url                        string        not null comment 'url del aviso; clave natural, una fila por url',
    portal                     string        comment 'host de la url sin www; los subdominios de mercadolibre se agrupan en mercadolibre',
    fecha_scraping             date          comment 'fecha en que se scrapeó la última versión del aviso (no es la fecha de publicación)',

    -- ubicación
    partido                    string        comment 'partido (municipio), mapeado desde zona con silver.ref_partidos',
    region                     string        comment 'capital federal, gba norte, gba oeste o gba sur',
    zona_original              string        comment 'zona tal como venía en bronze, para auditar el mapeo',
    calle                      string        comment 'calle; null si venía vacía',
    numero                     int           comment 'altura de la calle',

    -- operación y precio
    tipo_operacion             string        comment 'venta, alquiler, alquiler temporal u otro; null si no venía',
    moneda                     string        comment 'USD o ARS; null si la moneda no era reconocible',
    precio                     decimal(15,2) comment 'precio publicado; null si no era numérico o era <= 0',
    expensas                   decimal(15,2) comment 'expensas mensuales tal como vienen (0 se conserva)',

    -- características
    ambientes                  int           comment 'cantidad de ambientes; null si era <= 0',
    metros_cuadrados_totales   double        comment 'superficie total; null si era <= 0 o el centinela 2147483647',
    metros_cuadrados_cubiertos double        comment 'superficie cubierta; null si era <= 0 o el centinela 2147483647',
    piso                       int           comment 'piso entre 0 y 60; fuera de ese rango, null',
    antiguedad                 int           comment 'años entre 0 y 200; el centinela 999 pasa a null',
    cochera                    boolean       comment 'true si bronze traía 1; null si no se informa',
    estado                     string        comment 'a estrenar, excelente, muy bueno, bueno, reciclado, regular, a refaccionar o en construccion',
    orientacion                string        comment 'punto cardinal: norte, sur, este, oeste, noreste, noroeste, sureste o suroeste',
    posicion                   string        comment 'frente, contrafrente, lateral o interno',
    tipo_vendedor              string        comment 'dueno directo o inmobiliaria',

    -- historial del aviso
    primera_fecha_scraping     date          comment 'primera fecha en que apareció el aviso en bronze',
    veces_scrapeado            bigint        comment 'cantidad de versiones del aviso en bronze',

    -- calidad
    precio_por_m2              double        comment 'precio / metros_cuadrados_totales, redondeado a 2 decimales',
    flag_precio_outlier        boolean       comment 'precio fuera de P1-P99 de su moneda y tipo de operación (calculado sobre avisos deduplicados)',
    flag_moneda_sospechosa     boolean       comment 'venta en ARS con precio < 10.000.000: probable precio en USD mal etiquetado',
    flag_m2_inconsistente      boolean       comment 'm² cubiertos mayores que m² totales',
    apto_analisis_precio       boolean       comment 'true si tiene precio, moneda y operación válidos, no es outlier y no tiene moneda sospechosa',

    -- metadatos
    _source_table              string        comment 'tabla de origen',
    _processing_timestamp      timestamp     comment 'momento en que se construyó la tabla'
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
comment 'Avisos inmobiliarios en Silver: un aviso por fila (última versión por url), tipado, normalizado y con flags de calidad';



In [0]:
%sql
DESCRIBE TABLE EXTENDED bootcamp.silver.propiedades;


## Ejercicio 3.1: Ejecutar limpieza de Semana 2 (código starter)


In [0]:
%sql
create or replace temporary view propiedades_clean as (
select 
    case when precio rlike '^[^a-zA-Z]+$' then precio::double else null end as precio,
    case when ambientes rlike '^[^a-zA-Z]+$' then ambientes::double else null end as ambientes,
    case when metros_cuadrados_cubiertos rlike '^[^a-zA-Z]+$' then metros_cuadrados_cubiertos::double else null end as metros_cuadrados_cubiertos,
    case when metros_cuadrados_totales rlike '^[^a-zA-Z]+$' then metros_cuadrados_totales::double else null end as metros_cuadrados_totales,
    case when antiguedad rlike '^[^a-zA-Z]+$' then antiguedad::double else null end as antiguedad,
    tipo_de_operacion, id, ubicacion, numero, calle, expensas,
    orientacion_cardinal, orientacion_inmueble, piso, cochera, estado,
    tipo_vendedor, url, zona, fecha, hora, moneda
from bootcamp.bronze.properties_bronze
);
--veamos los prcentiles de precio
create or replace temporary view propiedades_clean_2 as (
with limites as (
    select 
        moneda, 
        tipo_de_operacion,
        percentile(precio,0.01) as p01,
        percentile(precio,0.99) as p99 
    from propiedades_clean
    where 1 = 1
    and precio > 0
    and moneda in ('ARS', 'USD')
    and tipo_de_operacion in ('venta', 'alquiler')
    group by all
)
select p.*
from propiedades_clean p
join limites l on p.tipo_de_operacion = l.tipo_de_operacion and p.moneda = l.moneda
where 1 = 1
and p.precio between l.p01 and l.p99
);
select * from propiedades_clean_2 limit 10


---
## Análisis estadístico de `propiedades_clean_2`: insumos para la limpieza Silver

Antes de poblar `bootcamp.silver.propiedades` se perfila la vista `propiedades_clean_2` en cuatro ejes: **moneda**, **zona**, **fecha** y **nulos**.

> Requiere haber ejecutado antes la celda que crea las vistas temporales `propiedades_clean` y `propiedades_clean_2` (Ejercicio 3.1).

| # | Eje | Pregunta |
|---|---|---|
| A | Panorama | ¿Cuántos registros y avisos únicos quedan tras el filtro de outliers? |
| B | Moneda | ¿Qué se pierde del Bronze y qué monedas están mal etiquetadas? |
| C | Zona | ¿Cuántos formatos hay y cuántos partidos reales esconden? |
| D | Fecha | ¿Qué representa la fecha y cómo se distribuye en el tiempo? |
| E | Nulos | ¿Qué columnas tienen nulos reales o disfrazados? |

### A. Panorama general

In [0]:
%sql
select
    count(*)                     as filas,
    count(distinct url)          as avisos_unicos,
    count(distinct url, fecha)   as aviso_fecha_unicos,
    count(*) - count(distinct url, fecha) as duplicados_misma_fecha,
    min(precio)                  as precio_min,
    max(precio)                  as precio_max
from propiedades_clean_2

### B. Moneda

La vista solo conserva **ARS** y **USD**. Hay que medir qué quedó afuera en Bronze y si las etiquetas de moneda son confiables.

In [0]:
%sql
select
    moneda,
    tipo_de_operacion,
    count(*)                                              as n,
    round(100 * count(*) / sum(count(*)) over (), 2)      as pct,
    percentile(precio, 0.25)                              as p25,
    percentile(precio, 0.50)                              as p50,
    percentile(precio, 0.75)                              as p75
from propiedades_clean_2
group by all
order by n desc

In [0]:
%sql
select moneda, count(*) as n
from bootcamp.bronze.properties_bronze
group by all
order by n desc

In [0]:
%sql
select
    count(*)                                                    as n,
    percentile(precio, array(0.10, 0.25, 0.50, 0.75, 0.90))     as p10_p25_p50_p75_p90
from propiedades_clean_2
where moneda = 'ARS' and tipo_de_operacion = 'venta'

In [0]:
%sql
select
    parse_url(url, 'HOST')                                                  as portal,
    count_if(tipo_de_operacion = 'alquiler')                                as alquileres,
    round(100 * count_if(tipo_de_operacion = 'alquiler' and moneda = 'USD')
          / nullif(count_if(tipo_de_operacion = 'alquiler'), 0), 1)         as pct_usd
from propiedades_clean_2
group by all
order by alquileres desc

In [0]:
%sql
select
    date_trunc('month', fecha::date)                        as mes,
    count(*)                                                as alquileres,
    round(100 * count_if(moneda = 'USD') / count(*), 1)     as pct_usd
from propiedades_clean_2
where tipo_de_operacion = 'alquiler'
group by all
order by mes

### C. Zona

Cada portal escribe la zona a su manera. Se mide la cardinalidad, se clasifican los formatos y se extrae el **partido** para saber cuántas zonas reales hay.

In [0]:
%sql
select
    case when zona like 'bsas-gba-%' then 'bsas-gba-<region>/<partido>'
         when zona like 'gba-zona-%' then 'gba-zona-<region>--<partido>'
         else '<partido>' end                   as formato,
    parse_url(url, 'HOST')                      as portal,
    count(*)                                    as n,
    count(distinct zona)                        as variantes
from propiedades_clean_2
group by all
order by n desc

In [0]:
%sql
with z as (
    select
        zona,
        case when zona rlike '/|--' then regexp_extract(zona, '(?:/|--)(.+)$', 1) else zona end as partido
    from propiedades_clean_2
)
select
    count(distinct zona)     as variantes_zona,
    count(distinct partido)  as partidos
from z

In [0]:
%sql
with z as (
    select
        zona,
        case when zona rlike '/|--' then regexp_extract(zona, '(?:/|--)(.+)$', 1) else zona end as partido
    from propiedades_clean_2
)
select
    partido,
    count(distinct zona)                  as variantes,
    count(*)                              as n,
    array_sort(collect_set(zona))         as zonas_originales
from z
group by partido
order by n desc

### D. Fecha

`fecha` y `hora` llegan como texto. Se valida el formato, el rango y cómo se reparte el volumen en el tiempo y entre portales.

In [0]:
%sql
select
    'fecha'                                  as campo,
    regexp_replace(fecha, '[0-9]', '9')      as patron,
    count(*)                                 as n,
    count(distinct fecha)                    as valores_distintos,
    min(fecha)                               as minimo,
    max(fecha)                               as maximo
from propiedades_clean_2
group by all
union all
select
    'hora',
    regexp_replace(hora, '[0-9]', '9'),
    count(*),
    count(distinct hora),
    min(hora),
    max(hora)
from propiedades_clean_2
group by all

In [0]:
%sql
select
    date_trunc('month', fecha::date)   as mes,
    parse_url(url, 'HOST')             as portal,
    count(*)                           as n
from propiedades_clean_2
group by all
order by mes, n desc

In [0]:
%sql
select
    veces,
    count(*) as avisos
from (
    select url, count(distinct fecha) as veces
    from propiedades_clean_2
    group by url
)
group by veces
order by veces

### E. Nulos

Se cuentan los **nulos reales** por columna y, por separado, los **nulos disfrazados**: valores centinela que un `is null` no detecta.

In [0]:
%sql
with conteo as (
    select stack(20,
        'precio',                     count_if(precio is null),
        'ambientes',                  count_if(ambientes is null),
        'metros_cuadrados_cubiertos', count_if(metros_cuadrados_cubiertos is null),
        'metros_cuadrados_totales',   count_if(metros_cuadrados_totales is null),
        'antiguedad',                 count_if(antiguedad is null),
        'tipo_de_operacion',          count_if(tipo_de_operacion is null),
        'id',                         count_if(id is null),
        'ubicacion',                  count_if(ubicacion is null),
        'numero',                     count_if(numero is null),
        'calle',                      count_if(calle is null),
        'expensas',                   count_if(expensas is null),
        'orientacion_cardinal',       count_if(orientacion_cardinal is null),
        'orientacion_inmueble',       count_if(orientacion_inmueble is null),
        'piso',                       count_if(piso is null),
        'cochera',                    count_if(cochera is null),
        'estado',                     count_if(estado is null),
        'tipo_vendedor',              count_if(tipo_vendedor is null),
        'url',                        count_if(url is null),
        'zona',                       count_if(zona is null),
        'moneda',                     count_if(moneda is null)
    ) as (columna, nulos)
    from propiedades_clean_2
)
select
    columna,
    nulos,
    round(100 * nulos / (select count(*) from propiedades_clean_2), 2) as pct_nulos
from conteo
order by nulos desc

In [0]:
%sql
select stack(6,
    'antiguedad = 999',                count_if(antiguedad = 999),
    'antiguedad nula o 999',           count_if(antiguedad is null or antiguedad = 999),
    'm² totales = 2147483647',         count_if(metros_cuadrados_totales >= 2147483647),
    'm² totales <= 0',                 count_if(metros_cuadrados_totales <= 0),
    'ambientes <= 0',                  count_if(ambientes <= 0),
    'm² cubiertos > m² totales',       count_if(metros_cuadrados_cubiertos > metros_cuadrados_totales)
) as (chequeo, filas)
from propiedades_clean_2

### ✅ Cómo abordar la limpieza en Silver

| Eje | Regla propuesta |
|---|---|
| Moneda | Mapear variantes a ISO (`USD`, `ARS`) y descartar o aislar monedas no soportadas. Marcar ventas en ARS con precio irreal como sospechosas de moneda mal etiquetada |
| Zona | Extraer `partido` del sufijo (`/` o `--`), derivar `region` del prefijo y completar la región de los registros que solo traen el partido con una tabla de mapeo |
| Fecha | Castear `fecha` a `date` como **fecha de scraping**, no de publicación. Descartar `hora` si no aporta variación |
| Duplicados | Un aviso aparece en varias fechas: deduplicar por `url` conservando la fecha más reciente |
| Nulos | Convertir centinelas (`999`, `2147483647`, valores `<= 0`) a `null` y eliminar columnas con nulos masivos |
| Comparaciones en el tiempo | Ponderar o separar por portal: los cambios de volumen y de mezcla de monedas pueden venir del scraping, no del mercado |

---
# Pipeline propio Bronze → Silver

Versión alternativa a `bronze_EDA` basada en el análisis estadístico anterior. Principios:

| Principio | Cómo se aplica |
|---|---|
| **Silver se construye desde Bronze** | No parte de `propiedades_clean_2`, que ya eliminó outliers |
| **Normalizar, no filtrar** | La zona se traduce a partido y región con una tabla de mapeo, sin descartar formatos |
| **Nunca inventar valores** | Lo inválido o centinela (`999`, `2147483647`, `<= 0`) pasa a `null`; no se rellena con `0`, `-1`, `999` ni `current_date()` |
| **Marcar en lugar de borrar** | Outliers e inconsistencias quedan como flags; el filtro lo decide cada análisis |
| **Un registro por aviso** | Se deduplica por `url` conservando el scraping más reciente |
| **Todo cuadra** | Bronze = Silver + versiones anteriores de un mismo aviso |

### Paso 1 · Tabla de referencia de partidos

Las 99 variantes de `zona` se reducen al partido con `regexp_extract`. Esta tabla traduce cada partido (o localidad) a su nombre estandarizado y su región, en lugar de un `case` de 90 líneas.

In [0]:
%sql
create or replace table bootcamp.silver.ref_partidos (
    zona_slug     string  comment 'Partido o localidad tal como aparece en la zona de Bronze',
    partido       string  comment 'Partido (municipio) estandarizado',
    region        string  comment 'capital federal, gba norte, gba oeste o gba sur',
    es_localidad  boolean comment 'true si el slug es una localidad y no un partido'
)
comment 'Tabla de referencia: zona de Bronze -> partido y región';

insert into bootcamp.silver.ref_partidos values
    ('capital-federal',     'capital federal',     'capital federal', false),
    ('escobar',             'escobar',             'gba norte', false),
    ('general-san-martin',  'general san martin',  'gba norte', false),
    ('jose-c-paz',          'jose c paz',          'gba norte', false),
    ('malvinas-argentinas', 'malvinas argentinas', 'gba norte', false),
    ('pilar',               'pilar',               'gba norte', false),
    ('san-fernando',        'san fernando',        'gba norte', false),
    ('san-isidro',          'san isidro',          'gba norte', false),
    ('san-miguel',          'san miguel',          'gba norte', false),
    ('tigre',               'tigre',               'gba norte', false),
    ('vicente-lopez',       'vicente lopez',       'gba norte', false),
    ('canuelas',            'canuelas',            'gba oeste', false),
    ('caseros',             'tres de febrero',     'gba oeste', true),
    ('castelar',            'moron',               'gba oeste', true),
    ('general-rodriguez',   'general rodriguez',   'gba oeste', false),
    ('hurlingham',          'hurlingham',          'gba oeste', false),
    ('ituzaingo',           'ituzaingo',           'gba oeste', false),
    ('la-matanza',          'la matanza',          'gba oeste', false),
    ('lujan',               'lujan',               'gba oeste', false),
    ('marcos-paz',          'marcos paz',          'gba oeste', false),
    ('merlo',               'merlo',               'gba oeste', false),
    ('moreno',              'moreno',              'gba oeste', false),
    ('moron',               'moron',               'gba oeste', false),
    ('tres-de-febrero',     'tres de febrero',     'gba oeste', false),
    ('almirante-brown',     'almirante brown',     'gba sur', false),
    ('avellaneda',          'avellaneda',          'gba sur', false),
    ('berazategui',         'berazategui',         'gba sur', false),
    ('berisso',             'berisso',             'gba sur', false),
    ('ensenada',            'ensenada',            'gba sur', false),
    ('esteban-echeverria',  'esteban echeverria',  'gba sur', false),
    ('ezeiza',              'ezeiza',              'gba sur', false),
    ('florencio-varela',    'florencio varela',    'gba sur', false),
    ('la-plata',            'la plata',            'gba sur', false),
    ('lanus',               'lanus',               'gba sur', false),
    ('lomas-de-zamora',     'lomas de zamora',     'gba sur', false),
    ('presidente-peron',    'presidente peron',    'gba sur', false),
    ('quilmes',             'quilmes',             'gba sur', false),
    ('san-vicente',         'san vicente',         'gba sur', false)

In [0]:
%sql
select * from bootcamp.silver.ref_partidos

In [0]:
%sql
select b.zona, count(*) as filas
from bootcamp.bronze.properties_bronze b
left anti join bootcamp.silver.ref_partidos r
    on r.zona_slug = case when b.zona rlike '/|--' then regexp_extract(b.zona, '(?:/|--)(.+)$', 1) else b.zona end
group by all

### Paso 2 · Vista `propiedades_tipadas`: tipos y normalización

Una fila por cada fila de Bronze, sin filtros. Se prueba con `select` antes de persistir.

| Columna | Regla |
|---|---|
| Numéricas | `try_cast` sobre el texto original; `<= 0` y `2147483647` → `null`; `piso` fuera de 0–60 y `antiguedad` fuera de 0–200 → `null` |
| `tipo_operacion` | `venta`, `alquiler` (incluye typos y anual), `alquiler temporal`; combinaciones y casos raros → `otro` |
| `moneda` | `USD`, `ARS` (incluye `ar`); el resto → `null` |
| `estado` | Sinónimos agrupados en 8 categorías; códigos (`-1`, `0`) y adjetivos (`luminoso`) → `null` |
| `orientacion` / `posicion` | Las dos columnas de Bronze mezclan puntos cardinales y posición: se separan por contenido |
| `cochera` | `1` → `true`; sin dato → `null` (no se asume `false`) |
| `tipo_vendedor` | `dueno directo` o `inmobiliaria` (los nombres de inmobiliarias se agrupan) |
| `portal` | Dominio de la `url`; los subdominios de MercadoLibre se unifican |

In [0]:
%sql
create or replace temporary view propiedades_tipadas as
with base as (
    select
        *,
        replace(lower(trim(estado)), '_', ' ')                as estado_norm,
        lower(trim(orientacion_cardinal))                     as ori_card,
        replace(lower(trim(orientacion_inmueble)), '_', ' ')  as ori_inm,
        try_cast(metros_cuadrados_totales as double)          as m2_tot,
        try_cast(metros_cuadrados_cubiertos as double)        as m2_cub
    from bootcamp.bronze.properties_bronze
)
select
    b.id                                                       as id_bronze,
    b.url,
    case when parse_url(b.url, 'HOST') like '%mercadolibre%' then 'mercadolibre'
         else regexp_replace(parse_url(b.url, 'HOST'), '^www\\.', '') end as portal,
    try_cast(b.fecha as date)                                  as fecha_scraping,

    -- ubicación
    r.partido,
    r.region,
    b.zona                                                     as zona_original,
    nullif(trim(b.calle), '')                                  as calle,
    try_cast(try_cast(b.numero as double) as int)              as numero,

    -- operación y precio
    case
        when lower(trim(b.tipo_de_operacion)) in ('venta') then 'venta'
        when lower(trim(b.tipo_de_operacion)) in ('alquiler', 'alquiler_anual', 'alquier', 'alquieler', 'alguilar') then 'alquiler'
        when lower(trim(b.tipo_de_operacion)) in ('alquiler_temporal', 'alquiler_temporario', 'temporario', 'alquiler_de_temporada') then 'alquiler temporal'
        when lower(trim(b.tipo_de_operacion)) in ('null', '') or b.tipo_de_operacion is null then null
        else 'otro'
    end                                                        as tipo_operacion,
    case
        when upper(trim(b.moneda)) = 'USD' then 'USD'
        when upper(trim(b.moneda)) in ('ARS', 'AR') then 'ARS'
    end                                                        as moneda,
    case when try_cast(b.precio as double) > 0
         then try_cast(try_cast(b.precio as double) as decimal(15,2)) end as precio,
    try_cast(try_cast(b.expensas as double) as decimal(15,2))  as expensas,

    -- características
    case when try_cast(b.ambientes as double) > 0
         then cast(try_cast(b.ambientes as double) as int) end as ambientes,
    case when b.m2_tot > 0 and b.m2_tot < 2147483647 then b.m2_tot end as metros_cuadrados_totales,
    case when b.m2_cub > 0 and b.m2_cub < 2147483647 then b.m2_cub end as metros_cuadrados_cubiertos,
    case when try_cast(b.piso as double) between 0 and 60
         then cast(try_cast(b.piso as double) as int) end      as piso,
    case when try_cast(b.antiguedad as double) between 0 and 200
         then cast(try_cast(b.antiguedad as double) as int) end as antiguedad,
    case when trim(b.cochera) = '1' then true end              as cochera,
    case
        when b.estado_norm in ('a estrenar', 'nuevo', 'terminada') then 'a estrenar'
        when b.estado_norm in ('excelente', 'impecable', 'perfecto', 'optimo') then 'excelente'
        when b.estado_norm in ('muy bueno', 'muy buena', 'muy buen estado') then 'muy bueno'
        when b.estado_norm in ('bueno', 'habitable', 'estandar', 'solida') then 'bueno'
        when b.estado_norm in ('reciclado', 'refaccionado', 'renovado') then 'reciclado'
        when b.estado_norm = 'regular' then 'regular'
        when b.estado_norm in ('a refaccionar', 'a reciclar') then 'a refaccionar'
        when b.estado_norm in ('en construccion', 'en obra', 'en pozo', 'en preventa') then 'en construccion'
    end                                                        as estado,
    -- cada portal mezcla orientación cardinal y posición en las dos columnas: se separan por contenido
    coalesce(
        case when b.ori_card in ('norte', 'n') then 'norte' when b.ori_card in ('sur', 's') then 'sur'
             when b.ori_card in ('este', 'e') then 'este' when b.ori_card in ('oeste', 'o') then 'oeste'
             when b.ori_card in ('noreste', 'ne') then 'noreste' when b.ori_card in ('noroeste', 'no') then 'noroeste'
             when b.ori_card in ('sureste', 'sudeste', 'se') then 'sureste'
             when b.ori_card in ('suroeste', 'sudoeste', 'so') then 'suroeste' end,
        case when b.ori_inm in ('norte', 'sur', 'este', 'oeste', 'noreste', 'noroeste') then b.ori_inm
             when b.ori_inm in ('sureste', 'sudeste') then 'sureste'
             when b.ori_inm in ('suroeste', 'sudoeste') then 'suroeste' end
    )                                                          as orientacion,
    coalesce(
        case when b.ori_inm in ('frente', 'a la calle', 'calle', 'externo', 'esquina') then 'frente'
             when b.ori_inm in ('contrafrente', 'fondo') then 'contrafrente'
             when b.ori_inm in ('lateral', 'interno') then b.ori_inm end,
        case when b.ori_card = 'frente' then 'frente' when b.ori_card = 'contrafrente' then 'contrafrente'
             when b.ori_card in ('lateral', 'interno') then b.ori_card end
    )                                                          as posicion,
    case
        when lower(trim(b.tipo_vendedor)) in ('dueno_directo', 'duenos_directos', 'propietario') then 'dueno directo'
        when b.tipo_vendedor is null or lower(trim(b.tipo_vendedor)) in ('null', '') then null
        else 'inmobiliaria'
    end                                                        as tipo_vendedor
from base b
left join bootcamp.silver.ref_partidos r
    on r.zona_slug = case when b.zona rlike '/|--' then regexp_extract(b.zona, '(?:/|--)(.+)$', 1) else b.zona end

In [0]:
%sql
select
    count(*)                                   as filas,
    count(distinct url)                        as avisos_unicos,
    count_if(partido is null)                  as sin_partido,
    count_if(fecha_scraping is null)           as sin_fecha,
    count_if(precio is null)                   as sin_precio,
    count_if(moneda is null)                   as sin_moneda,
    count_if(tipo_operacion is null)           as sin_tipo_operacion,
    count_if(tipo_operacion = 'otro')          as tipo_operacion_otro
from propiedades_tipadas

### Paso 3 · Tabla `bootcamp.silver.propiedades`

- **Deduplicación:** un registro por `url`, el del scraping más reciente. Se conservan `primera_fecha_scraping` y `veces_scrapeado`.
- **Flags de calidad:**
  - `flag_precio_outlier`: fuera de P1–P99 de su moneda × operación.
  - `flag_moneda_sospechosa`: venta en ARS por menos de 10 millones (≈ precio en USD mal etiquetado).
  - `flag_m2_inconsistente`: m² cubiertos > m² totales.
  - `apto_analisis_precio`: precio, moneda y operación válidos, dentro de P1–P99 y sin moneda sospechosa.

In [0]:
%sql
insert overwrite bootcamp.silver.propiedades
with
historial as (
    select
        *,
        row_number()               over (partition by url order by fecha_scraping desc, id_bronze desc) as rn,
        min(fecha_scraping)        over (partition by url) as primera_fecha_scraping,
        count(*)                   over (partition by url) as veces_scrapeado
    from propiedades_tipadas
),
ultimo as (
    select * except (rn) from historial where rn = 1
),
limites as (
    select
        moneda,
        tipo_operacion,
        percentile(precio, 0.01) as p01,
        percentile(precio, 0.99) as p99
    from ultimo
    where precio is not null and moneda is not null
    group by all
)
select
    xxhash64(u.url)                                                 as propiedad_id,
    u.*,
    case when u.precio is not null and u.metros_cuadrados_totales is not null
         then round(u.precio / u.metros_cuadrados_totales, 2) end   as precio_por_m2,

    u.precio is not null and (u.precio < l.p01 or u.precio > l.p99) as flag_precio_outlier,
    u.moneda = 'ARS' and u.tipo_operacion = 'venta' and u.precio < 10000000
                                                                    as flag_moneda_sospechosa,
    u.metros_cuadrados_cubiertos > u.metros_cuadrados_totales       as flag_m2_inconsistente,
    coalesce(
        u.precio is not null
        and u.moneda is not null
        and u.tipo_operacion in ('venta', 'alquiler', 'alquiler temporal')
        and u.precio between l.p01 and l.p99
        and not (u.moneda = 'ARS' and u.tipo_operacion = 'venta' and u.precio < 10000000),
        false
    )                                                               as apto_analisis_precio,

    'bootcamp.bronze.properties_bronze'                             as _source_table,
    current_timestamp()                                             as _processing_timestamp
from ultimo u
left join limites l
    on u.moneda = l.moneda and u.tipo_operacion = l.tipo_operacion

### Paso 4 · Validación de Silver

In [0]:
%sql
select
    (select count(*) from bootcamp.bronze.properties_bronze)               as filas_bronze,
    (select count(distinct url) from bootcamp.bronze.properties_bronze)    as avisos_unicos_bronze,
    (select count(*) from bootcamp.silver.propiedades)                     as filas_silver,
    (select count(*) from bootcamp.bronze.properties_bronze)
      - (select count(*) from bootcamp.silver.propiedades)                 as versiones_anteriores_descartadas,
    (select count(*) - count(distinct url) from bootcamp.silver.propiedades) as urls_repetidas_en_silver

In [0]:
%sql
select
    count_if(antiguedad = 999)                          as antiguedad_999,
    count_if(metros_cuadrados_totales >= 2147483647)    as m2_centinela,
    count_if(precio <= 0)                               as precio_no_positivo,
    count_if(ambientes <= 0)                            as ambientes_no_positivos,
    count_if(estado in ('-1', '-2', '0'))               as estado_centinela,
    count_if(fecha_scraping > current_date())           as fechas_futuras
from bootcamp.silver.propiedades

In [0]:
%sql
select
    region,
    count(*)                                              as avisos,
    count(distinct partido)                               as partidos,
    round(100 * count(*) / sum(count(*)) over (), 2)      as pct
from bootcamp.silver.propiedades
group by all
order by avisos desc

In [0]:
%sql
select
    tipo_operacion,
    moneda,
    count(*)                              as avisos,
    count_if(flag_precio_outlier)         as outliers_precio,
    count_if(flag_moneda_sospechosa)      as moneda_sospechosa,
    count_if(flag_m2_inconsistente)       as m2_inconsistente,
    count_if(apto_analisis_precio)        as aptos_analisis_precio
from bootcamp.silver.propiedades
group by all
order by avisos desc

In [0]:
%sql
with conteo as (
    select stack(17,
        'partido',                     count_if(partido is null),
        'tipo_operacion',              count_if(tipo_operacion is null),
        'moneda',                      count_if(moneda is null),
        'precio',                      count_if(precio is null),
        'expensas',                    count_if(expensas is null),
        'ambientes',                   count_if(ambientes is null),
        'metros_cuadrados_totales',    count_if(metros_cuadrados_totales is null),
        'metros_cuadrados_cubiertos',  count_if(metros_cuadrados_cubiertos is null),
        'piso',                        count_if(piso is null),
        'antiguedad',                  count_if(antiguedad is null),
        'cochera',                     count_if(cochera is null),
        'estado',                      count_if(estado is null),
        'orientacion',                 count_if(orientacion is null),
        'posicion',                    count_if(posicion is null),
        'tipo_vendedor',               count_if(tipo_vendedor is null),
        'calle',                       count_if(calle is null),
        'precio_por_m2',               count_if(precio_por_m2 is null)
    ) as (columna, nulos)
    from bootcamp.silver.propiedades
)
select columna, nulos, round(100 * nulos / (select count(*) from bootcamp.silver.propiedades), 2) as pct_nulos
from conteo
order by nulos desc

In [0]:
%sql
select
    tipo_operacion,
    moneda,
    count(*)                         as avisos,
    percentile(precio, 0.5)          as mediana_precio,
    percentile(precio_por_m2, 0.5)   as mediana_precio_m2
from bootcamp.silver.propiedades
where apto_analisis_precio
group by all
order by avisos desc